# Chapter 5 (part 1) — Naive Bayes and LDA: Two Quick Churn Baselines

Covers: Naive Bayes → Discriminant Analysis (LDA), on the **churn** data, plus your first feature-engineering step.

**Chapter 5 project question: who will churn, and who should a limited retention budget target?** The data: IBM's Telco customer churn set, 7,043 customers with their contract, tenure, monthly and total charges, and which services they pay for; 26.5% churned. Every Chapter 5–6 notebook uses the same fixed train/test split from `load_churn_split()` (4,930 / 2,113 customers), so every model you build is compared fairly against the ones before it.

This notebook is a short warm-up (~40 min): you build a few features, then two fast baselines that later models must beat. Accuracy at a 0.5 cutoff is the only scorecard for now; notebook 15 replaces it with metrics built for this problem.

**How to work each section:** write your prediction **before** running anything, do the code, then write 2-3 bullets on what matched and what didn't. Feature engineering is yours; the encoding and scaling plumbing is pre-filled. The exam for this notebook is appended to `reviews/R01_regression-diagnostics.ipynb` (Q6–Q8).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import CategoricalNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

pd.set_option("display.max_columns", None)

from mlstats.data import load_churn_split

train, test = load_churn_split()   # fixed stratified split; customerID already dropped
y_train, y_test = train["churned"].to_numpy(), test["churned"].to_numpy()

ADDONS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]


def report(p, y, label):
    # pre-filled scorecard: accuracy at a 0.5 cutoff vs. predicting "everyone stays"
    pred = p >= 0.5
    print(f"{label:28s} accuracy {np.mean(pred == y):.3f} (everyone-stays baseline {1 - y.mean():.3f}) | "
          f"churners caught {pred[y == 1].mean():.1%} | customers flagged {pred.sum()} of {len(y)}")


def confidence_table(p, y):
    # pre-filled: when the model says X%, how often do those customers actually churn?
    bands = pd.cut(p, [0, 0.1, 0.5, 0.9, 1.0], include_lowest=True)
    return (pd.DataFrame({"band": bands, "p": p, "churned": y})
              .groupby("band", observed=True)
              .agg(share_of_customers=("p", lambda s: len(s) / len(p)), mean_predicted=("p", "mean"), actual_churn_rate=("churned", "mean"))
              .round(3))


print(train.shape, test.shape, "| train churn rate:", round(y_train.mean(), 4))

In [ ]:
train.head()

## 1. Build Your Features

Covers: *supplementary* (feature engineering; the book assumes it is done). A model only sees the columns you hand it, so turning raw columns into ones that carry the signal more directly is often the biggest lever you have. Three features, each from a relationship in the raw data:

- **`n_addons`**: how many of the six add-on services (`ADDONS`) the customer has. A fiber customer with TechSupport and OnlineSecurity has 2. Six correlated yes/no columns become one count.
- **`tenure_band`**: tenure binned into `0-12`, `13-24`, `25-48`, `49-72` months, so Naive Bayes (which needs categories) can use it.
- **`charge_drift`**: current monthly bill minus lifetime average monthly bill. Numeric picture: 10 months in with `TotalCharges` = $700 means a $70 average; if `MonthlyCharges` is now $80, drift = +$10 (a price increase or an added service). Drift = −$10 means a downgrade or discount.

Two rules that matter in every future notebook: write one function and apply it to train and test alike, and compute nothing from the test set (fixed bin edges here are fine; edges taken from quantiles would have to come from train only). Add a feature of your own if you like; append its name to the column lists in sections 2–3 to test it.

**Predict first:** (a) as `n_addons` goes from 0 to 6, does the churn rate fall steadily, or rise first and then fall? (b) Do customers whose bill drifted **up** by more than $2 churn more than those whose bill drifted **down** by more than $2 (by over 5 points), about the same, or less?

*Tools for this question:*
- `(df[ADDONS] == "Yes").sum(axis=1)` → count of "Yes" per row. Gotcha: "No internet service" is not "Yes".
- `pd.cut(x, bins=[0, 12, 24, 48, 72], labels=["0-12", "13-24", "25-48", "49-72"], include_lowest=True)` → categorical. Without `include_lowest=True`, tenure 0 becomes NaN.
- `TotalCharges / tenure` at tenure 0 gives NaN or inf (11 brand-new customers). Define their drift as 0, e.g. with `np.where(df["tenure"] > 0, ..., df["MonthlyCharges"])` for the average.
- Start from `out = df.copy()` and return `out`; never modify the input in place.

_Prediction (one line, before running anything):_

In [ ]:
def add_features(df):
    """Return a copy of df with three new columns: n_addons, tenure_band, charge_drift."""
    ...

In [ ]:
train_f = ...  # add_features applied to train
test_f = ...   # add_features applied to test

In [ ]:
# pre-filled: sanity checks + churn rate by each new feature (consumes your train_f / test_f)
for d in (train_f, test_f):
    assert {"n_addons", "tenure_band", "charge_drift"} <= set(d.columns), "missing a feature"
    assert d["tenure_band"].notna().all(), "tenure_band has NaN (include_lowest?)"
    assert np.isfinite(d["charge_drift"]).all(), "charge_drift has NaN/inf (tenure == 0?)"

print(train_f.groupby("tenure_band", observed=True)["churned"].agg(churn_rate="mean", customers="size").round(3), "\n")
print(train_f.groupby("n_addons")["churned"].agg(churn_rate="mean", customers="size").round(3), "\n")
drift_bucket = pd.cut(train_f["charge_drift"], [-np.inf, -2, 2, np.inf], labels=["down > $2", "within $2", "up > $2"])
print(train_f.groupby(drift_bucket, observed=True)["churned"].agg(churn_rate="mean", customers="size").round(3))

_What matched, what surprised you, and why (2-3 bullets):_

## 2. Naive Bayes

Covers: *Naive Bayes* (book). The exact Bayesian answer to "how likely is churn for a customer with these traits?" is the churn rate among past customers with **exactly** those traits. With 16 categorical columns, few customers match exactly, so Naive Bayes multiplies one-clue-at-a-time evidence instead. Numeric picture: if 80% of churners but 40% of stayers have clue A, seeing A doubles the churn odds. If clue B also has a 2-to-1 ratio, the model doubles the odds again, even when B mostly repeats A. That is the "naive" part: it assumes the clues are independent within each class, so correlated clues get counted more than once.

(a) **By hand, once:** for the profile *Contract = Month-to-month, InternetService = Fiber optic, PaymentMethod = Electronic check*, compute `p_naive` from `train_f` with the formula below, and `p_exact` = the churn rate among training customers who match all three.
(b) **Library:** fit `CategoricalNB` twice: on all the categorical columns plus your two categorical features (`cols_all`), and on a lean set where your `n_addons` replaces the six add-on columns (`cols_lean`).

**Predict first:** (a) will `p_naive` come out higher than `p_exact`, about equal (within 3 points), or lower? (b) Will the all-columns model beat the 73.5% everyone-stays baseline on accuracy, and will the lean model do better or worse than it?

*Tools for this question:*
- Naive posterior: P(churn | x) = P(churn)·∏ P(xⱼ | churn) / [P(churn)·∏ P(xⱼ | churn) + P(stay)·∏ P(xⱼ | stay)].
- `(train_f.loc[train_f["churned"] == 1, "Contract"] == "Month-to-month").mean()` → P(Contract = Month-to-month | churn).
- `CategoricalNB().fit(X, y)`; `.predict_proba(X)[:, 1]` → P(churn). Default `alpha=1.0` is Laplace smoothing (adds 1 to every count so a category never seen with churn doesn't zero out the whole product). `X` must be non-negative integer codes; the pre-filled `OrdinalEncoder` cell does that.

_Prediction (one line, before running anything):_

In [ ]:
p_naive = ...  # naive-Bayes P(churn) for the profile, from train_f counts
p_exact = ...  # churn rate among train_f customers matching all three traits

In [ ]:
# pre-filled
print("naive Bayes:", round(p_naive, 3), "| exact:", round(p_exact, 3))

In [ ]:
# pre-filled: categorical columns -> integer codes (encoder fit on train only)
base_cats = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines",
             "InternetService", *ADDONS, "Contract", "PaperlessBilling", "PaymentMethod"]
cols_all = base_cats + ["tenure_band", "n_addons"]
cols_lean = ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling", "SeniorCitizen", "tenure_band", "n_addons"]


def encode(cols):
    enc = OrdinalEncoder().fit(train_f[cols].astype(str))
    return enc.transform(train_f[cols].astype(str)), enc.transform(test_f[cols].astype(str))


Xtr_all, Xte_all = encode(cols_all)
Xtr_lean, Xte_lean = encode(cols_lean)
print(len(cols_all), "columns vs", len(cols_lean), "columns")

In [ ]:
nb_all = ...     # CategoricalNB fit on Xtr_all, y_train
p_nb_all = ...   # its P(churn) on Xte_all
nb_lean = ...    # CategoricalNB fit on Xtr_lean, y_train
p_nb_lean = ...  # its P(churn) on Xte_lean

In [ ]:
# pre-filled: consumes p_nb_all, p_nb_lean
report(p_nb_all, y_test, "Naive Bayes, all columns")
report(p_nb_lean, y_test, "Naive Bayes, lean")
print("\nAll-columns model — does 'P = 97%' mean 97%?")
confidence_table(p_nb_all, y_test)

_What matched, what surprised you, and why (2-3 bullets):_

## 3. Linear Discriminant Analysis

Covers: *Discriminant Analysis* (book). LDA works on numeric features. Numeric picture with one feature: churners average about 18 months of tenure, stayers about 38, and tenure's spread within each group is about 24 months, so the two groups sit roughly 0.8 standard deviations apart. LDA finds the **weighted sum** of all the features that pushes the two group means furthest apart relative to the spread inside each group, assuming each group is a bell-shaped cloud with the same shape. With standardized inputs (pre-filled, scaler fit on train only), a weight says how far the score moves per 1-SD change in that feature, holding the others fixed, exactly like a coefficient in notebook 10.

Fit LDA on six numeric features, two of them yours: `tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`, `n_addons`, `charge_drift`.

**Predict first:** (a) will LDA beat the lean Naive Bayes on accuracy, and will it catch more or fewer churners? (b) Which feature gets the largest absolute weight: `tenure`, `MonthlyCharges`, or `TotalCharges`?

*Tools for this question:*
- `LinearDiscriminantAnalysis().fit(X, y)`; `.predict_proba(X)[:, 1]` → P(churn).
- `.coef_[0]` → one weight per feature, in the order of `num_cols` (comparable across features because the inputs are standardized).

_Prediction (one line, before running anything):_

In [ ]:
# pre-filled: standardize numeric features (scaler fit on train only)
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "n_addons", "charge_drift"]
scaler = StandardScaler().fit(train_f[num_cols])
Xtr_num, Xte_num = scaler.transform(train_f[num_cols]), scaler.transform(test_f[num_cols])

In [ ]:
lda = ...    # LinearDiscriminantAnalysis fit on Xtr_num, y_train
p_lda = ...  # its P(churn) on Xte_num

In [ ]:
# pre-filled: consumes p_lda, lda, and section 2's p_nb_lean
report(p_nb_lean, y_test, "Naive Bayes, lean")
report(p_lda, y_test, "LDA, 6 numeric")
pd.Series(lda.coef_[0], index=num_cols, name="standardized weight").sort_values(key=abs, ascending=False).round(3)

_What matched, what surprised you, and why (2-3 bullets):_

**Where this leaves the project question:** in one or two sentences, which of today's baselines would you hand the retention team, and what would you warn them about in Naive Bayes's probabilities?